# **Import Library**

In [10]:
%pip install wandb timm -q

import os
import copy
import time
import numpy as np
import matplotlib
print(matplotlib.__version__)
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import wandb

from dotenv import load_dotenv
load_dotenv()

# Ambil API key dari environment variable
wandb_api_key = os.getenv("WANDB_API_KEY")

# Login ke wandb
wandb.login(key=wandb_api_key)

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import random

from torchvision import transforms, datasets
import timm   # PERBAIKAN: ganti torchvision.models.resnet50 -> timm (untuk ViT)

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

SAVE_DIR = "models"
os.makedirs(SAVE_DIR, exist_ok=True)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Note: you may need to restart the kernel to use updated packages.
3.11.1


# **Dataset Path**

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
    
TRAIN_DIR = r"D:\Devianest_SkripsiTest\train"
TEST_DIR  = r"D:\Devianest_SkripsiTest\test"

cuda
2.11.0+cu128
True
NVIDIA GeForce RTX 4060


# **Train Augmentation**

In [12]:

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    #transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    #transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(10),
    #transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),  # PERBAIKAN: aktifkan, ringan saja
    transforms.RandomResizedCrop(224, scale=(0.8,1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    #transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])


# **Validation Transform**

In [13]:
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    #transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# **Load Filepaths**

In [14]:
classes = sorted(os.listdir(TRAIN_DIR))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

num_classes = len(classes)

filepaths = []
labels    = []

for label in classes:
    class_path = os.path.join(TRAIN_DIR, label)
    for img in os.listdir(class_path):
        filepaths.append(os.path.join(class_path, img))
        labels.append(class_to_idx[label])

print("Total Images :", len(filepaths))
print("Classes      :", classes)

Total Images : 15557
Classes      : ['Acne and Rosacea Photos', 'Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions', 'Atopic Dermatitis Photos', 'Bullous Disease Photos', 'Cellulitis Impetigo and other Bacterial Infections', 'Eczema Photos', 'Exanthems and Drug Eruptions', 'Hair Loss Photos Alopecia and other Hair Diseases', 'Herpes HPV and other STDs Photos', 'Light Diseases and Disorders of Pigmentation', 'Lupus and other Connective Tissue diseases', 'Melanoma Skin Cancer Nevi and Moles', 'Nail Fungus and other Nail Disease', 'Poison Ivy Photos and other Contact Dermatitis', 'Psoriasis pictures Lichen Planus and related diseases', 'Scabies Lyme Disease and other Infestations and Bites', 'Seborrheic Keratoses and other Benign Tumors', 'Systemic Disease', 'Tinea Ringworm Candidiasis and other Fungal Infections', 'Urticaria Hives', 'Vascular Tumors', 'Vasculitis Photos', 'Warts Molluscum and other Viral Infections']


# **Dataset Class**

In [15]:
class SkinDataset(Dataset):

    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# **Early Stopping & K-Fold**

In [16]:
class EarlyStopping:

    def __init__(self, patience=5):
        self.patience  = patience
        self.best_f1   = -np.inf
        self.counter   = 0

    def step(self, val_f1):
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# **Wandb**

In [17]:
BATCH_SIZE      = 16        
EPOCHS          = 100
EXPERIMENT_NAME = "EXP02_ViT_Base16_ECA"  

# ── HYPERPARAMETER KANDIDAT TERBAIK ──────────────────────────────────────────
LR               = 1e-4         
WEIGHT_DECAY     = 0.05            
DROP_OUT         = 0.2
UNFROZEN_LAYERS  = "last_4_blocks"
AUGMENTATION_STRENGTH = "medium"
LABEL_SMOOTHING = 0.1


USE_ECA          = True
ECA_K_SIZE       = 3  

WARMUP_EPOCHS    = 5

run = wandb.init(
    project = "SkinDisease-ViT",
    entity  = "devianestnarendra_Team",  # Ganti dengan username WandB Anda
    name    = EXPERIMENT_NAME,
    config  = {
        "architecture"   : "ViT-Base/16 (timm: vit_base_patch16_224)",
        "n_folds"        : 5,
        "epochs"         : EPOCHS,
        "batch_size"     : BATCH_SIZE,
        "optimizer"      : "AdamW",
        "lr"             : LR,
        "weight_decay"   : WEIGHT_DECAY,
        "Drop_Out"       : DROP_OUT,
        "unfrozen_layers": UNFROZEN_LAYERS,
        "augmentation_strength": AUGMENTATION_STRENGTH,
        "label_smoothing": LABEL_SMOOTHING,
        "use_eca"        : USE_ECA,
        "eca_k_size"     : ECA_K_SIZE,
        "warmup_epochs"  : WARMUP_EPOCHS,
        "lr_scheduler"   : "linear_warmup_cosine_decay",
        "checkpoint_criteria": "best_val_f1",
    }
)

print(f"WandB Run : {run.name}")
print(f"URL       : {run.url}")
wandb.run.log_code(".")


epoch,▁
fold_1/accuracy,▁
fold_1/f1_score,▁
fold_1/lr,▁
fold_1/precision,▁
fold_1/recall,▁
fold_1/train_loss,▁
fold_1/val_loss,▁
epoch,1
fold_1/accuracy,0.28021
fold_1/f1_score,0.27339


WandB Run : EXP02_ViT_Base16_ECA
URL       : https://wandb.ai/devianestnarendra_Team/SkinDisease-ViT/runs/2o0g5od6


<Artifact source-SkinDisease-ViT-d__Devianest_SkripsiTest_exp02-vit-b-16-eca.ipynb>

wandb: WARNING Artifact "source-SkinDisease-ViT-d__Devianest_SkripsiTest_exp02-vit-b-16-eca.ipynb" already exists with the same content. No new version will be created.


# **Training Loop**

In [18]:

class ECAAttention(nn.Module):
    def __init__(self, channels, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, C, H, W)
        y = self.avg_pool(x)                   
        y = y.squeeze(-1).transpose(-1, -2)        
        y = self.conv(y)                            
        y = y.transpose(-1, -2).unsqueeze(-1)        
        y = self.sigmoid(y)
        return x * y.expand_as(x)


class ECAPatchEmbed(nn.Module):
    def __init__(self, original_patch_embed, eca_k_size=3):
        super().__init__()
        self.proj = original_patch_embed.proj  
        self.norm = original_patch_embed.norm   
        self.eca = ECAAttention(self.proj.out_channels, k_size=eca_k_size)

    def forward(self, x):
        x = self.proj(x)                      
        x = self.eca(x)                             
        x = x.flatten(2).transpose(1, 2)          
        x = self.norm(x)
        return x



def apply_freeze_strategy(model, strategy: str):

    for param in model.parameters():
        param.requires_grad = False

    if strategy == "head_only":
        for param in model.head.parameters():
            param.requires_grad = True

    elif strategy == "last_2_blocks":
        for block in model.blocks[-2:]:
            for param in block.parameters():
                param.requires_grad = True
        for param in model.norm.parameters():
            param.requires_grad = True
        for param in model.head.parameters():
            param.requires_grad = True

    elif strategy == "last_4_blocks":
        for block in model.blocks[-4:]:
            for param in block.parameters():
                param.requires_grad = True
        for param in model.norm.parameters():
            param.requires_grad = True
        for param in model.head.parameters():
            param.requires_grad = True

    elif strategy == "all":
        for param in model.parameters():
            param.requires_grad = True

    else:
        raise ValueError(f"Unknown freeze strategy: {strategy}")


    if hasattr(model.patch_embed, "eca"):
        for param in model.patch_embed.eca.parameters():
            param.requires_grad = True

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f"  Freeze strategy   : {strategy}")
    print(f"  Trainable params  : {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)")

    return model



def get_lr_at_epoch(epoch, total_epochs, base_lr, warmup_epochs):
    import math
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
        return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


fold_results = []
fold_accuracies    = []
fold_precision     = []
fold_recall        = []
fold_f1            = []
all_fold_best_paths = []

all_train_losses = {}
all_val_losses   = {}


for fold, (train_idx, val_idx) in enumerate(skf.split(filepaths, labels)):

    # if fold < 4:
    #     continue

    print(f"\n{'='*50}")
    print(f"  FOLD {fold + 1} / 5")
    print(f"{'='*50}")

    best_val_loss   = np.inf
    best_train_loss = np.inf
    best_val_f1     = -np.inf   
    best_model_path = None

 # ── SPLIT ─────────────────────────────────────────────────────────────────
    train_files  = [filepaths[i] for i in train_idx]
    train_labels = [labels[i]    for i in train_idx]
    val_files    = [filepaths[i] for i in val_idx]
    val_labels   = [labels[i]    for i in val_idx]

    # ── DATALOADER ────────────────────────────────────────────────────────────
    train_loader = DataLoader(
        SkinDataset(train_files, train_labels, transform=train_tf),
        batch_size=BATCH_SIZE, shuffle=True,  num_workers=0
    )
    val_loader = DataLoader(
        SkinDataset(val_files, val_labels, transform=eval_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    # ── MODEL ─────────────────────────────────────────────────────────────────

    model = timm.create_model(
        "vit_base_patch16_224",
        pretrained=True,
        num_classes=num_classes,
        drop_rate=DROP_OUT,
        drop_path_rate = 0.1
    )


    if USE_ECA:
        model.patch_embed = ECAPatchEmbed(
            model.patch_embed,
            eca_k_size=ECA_K_SIZE
        )

    model = apply_freeze_strategy(model, UNFROZEN_LAYERS)

    model = model.to(device)


    # ── LOSS / OPTIMIZER / SCHEDULER ─────────────────────────────────────────
    class_counts  = np.bincount(train_labels)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device),
        label_smoothing=LABEL_SMOOTHING  
    )

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    early_stopping = EarlyStopping(patience=5)
    best_model_wts = copy.deepcopy(model.state_dict())

    train_losses = []
    val_losses   = []

    # ── EPOCH LOOP ────────────────────────────────────────────────────────────
    for epoch in range(EPOCHS):


        current_lr = get_lr_at_epoch(epoch, EPOCHS, LR, WARMUP_EPOCHS)
        for param_group in optimizer.param_groups:
            param_group['lr'] = current_lr

        print(f"\nEpoch {epoch + 1}/{EPOCHS}  (LR: {current_lr:.2e})")

        # TRAIN
        model.train()
        train_loss = 0
        for images, targets in tqdm(train_loader, desc="Train"):
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # VALIDATION
        model.eval()
        val_loss = 0
        preds, trues = [], []
        with torch.no_grad():
            for images, targets in tqdm(val_loader, desc="Val"):
                images, targets = images.to(device), targets.to(device)
                outputs   = model(images)
                val_loss += criterion(outputs, targets).item()
                preds.extend(outputs.argmax(1).cpu().numpy())
                trues.extend(targets.cpu().numpy())

        # METRICS
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss   / len(val_loader)

        acc       = accuracy_score(trues, preds)
        precision = precision_score(trues, preds, average='weighted', zero_division=0)
        recall    = recall_score(trues, preds, average='weighted', zero_division=0)
        f1        = f1_score(trues, preds, average='weighted', zero_division=0)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Train Loss : {avg_train_loss:.4f} | Val Loss  : {avg_val_loss:.4f}")
        print(f"Accuracy   : {acc:.4f}  | Precision : {precision:.4f}")
        print(f"Recall     : {recall:.4f}  | F1 Score  : {f1:.4f}")

        wandb.log({
            "epoch"                    : epoch + 1,

            f"fold_{fold+1}/train_loss": avg_train_loss,
            f"fold_{fold+1}/val_loss"  : avg_val_loss,

            f"fold_{fold+1}/accuracy"  : acc,
            f"fold_{fold+1}/precision" : precision,
            f"fold_{fold+1}/recall"    : recall,
            f"fold_{fold+1}/f1_score"  : f1,

            f"fold_{fold+1}/lr"        : optimizer.param_groups[0]['lr'],

        })



        # SAVE BEST MODEL

        if f1 > best_val_f1:
            best_val_f1     = f1
            best_val_loss   = avg_val_loss
            best_train_loss = avg_train_loss

            save_path      = os.path.join(SAVE_DIR, f"model_fold_{fold + 1}.pth")
            
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_loss"        : avg_val_loss,
                "f1"              : f1,
                "fold"            : fold + 1
            }, save_path)
            best_model_path = save_path
            best_model_wts  = copy.deepcopy(model.state_dict())
            print(f"  ✓ Model saved (best val_f1: {best_val_f1:.4f}) → {save_path}")

        
        if early_stopping.step(f1):
            print("Early Stopping Triggered")
            break

    # ── SIMPAN HISTORY ────────────────────────────────────────────────────────
    all_train_losses[fold + 1] = train_losses
    all_val_losses[fold + 1]   = val_losses

    # ── PLOT LOSS CURVE PER FOLD ──────────────────────────────────────────────
    epochs_ran = range(1, len(train_losses) + 1)
    fig, ax    = plt.subplots(figsize=(8, 5))
    ax.plot(epochs_ran, train_losses, label='Train Loss', marker='o', markersize=3)
    ax.plot(epochs_ran, val_losses,   label='Val Loss',   marker='o', markersize=3)
    ax.set_title(f'Fold {fold + 1} — Loss Curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    loss_curve_path = os.path.join(SAVE_DIR, f"Fold_{fold + 1}_Loss_Curve.png")

    plt.savefig(loss_curve_path, dpi=150, bbox_inches="tight")

    wandb.log({
        f"Loss_Curve/Fold_{fold+1}": wandb.Image(loss_curve_path)
    })

    plt.close(fig)

    print(f"✓ Loss curve saved → {loss_curve_path}")

    # ── UPLOAD MODEL ARTIFACT ─────────────────────────────────────────────────
    if best_model_path:
        artifact = wandb.Artifact(name=f"model-fold-{fold+1}", type="model")
        artifact.add_file(best_model_path)
        wandb.log_artifact(artifact)
        all_fold_best_paths.append(best_model_path)

    # ── FINAL EVALUATION FOLD (pakai best model) ──────────────────────────────
    if best_model_path:
        model.load_state_dict(
            torch.load(best_model_path, map_location=device)["model_state_dict"]
        )

    model.eval()
    final_preds, final_trues = [], []
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            final_preds.extend(model(images).argmax(1).cpu().numpy())
            final_trues.extend(targets.cpu().numpy())

    print("\nClassification Report")
    print(classification_report(final_trues, final_preds, target_names=classes, zero_division=0))

    fold_acc  = accuracy_score(final_trues, final_preds)
    fold_prec = precision_score(final_trues, final_preds, average='weighted', zero_division=0)
    fold_rec  = recall_score(final_trues, final_preds, average='weighted', zero_division=0)
    fold_f1_  = f1_score(final_trues, final_preds, average='weighted', zero_division=0)

    fold_accuracies.append(fold_acc)
    fold_precision.append(fold_prec)
    fold_recall.append(fold_rec)
    fold_f1.append(fold_f1_)

    fold_results.append({
        "Fold": fold + 1,
        "Train_Loss": best_train_loss,
        "Val_Loss": best_val_loss,
        "Accuracy": fold_acc,
        "Precision": fold_prec,
        "Recall": fold_rec,
        "F1": fold_f1_
})

    # ── WANDB LOG FINAL METRICS FOLD ─────────────────────────────────────────

    wandb.log({
        f"fold_{fold+1}/final_accuracy" : fold_acc,
        f"fold_{fold+1}/final_precision": fold_prec,
        f"fold_{fold+1}/final_recall"   : fold_rec,
        f"fold_{fold+1}/final_f1"       : fold_f1_,

        f"fold_{fold+1}/confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=final_trues,
            preds=final_preds,
            class_names=classes
        )
    })

    print(f"\nFold {fold+1} selesai — Acc: {fold_acc:.4f} | F1: {fold_f1_:.4f}")

    del model, optimizer, best_model_wts
    torch.cuda.empty_cache()


results_df = pd.DataFrame(fold_results)

results_df.loc[len(results_df)] = {
    "Fold": "Mean",
    "Train_Loss": results_df["Train_Loss"].mean(),
    "Val_Loss": results_df["Val_Loss"].mean(),
    "Accuracy": np.mean(fold_accuracies),
    "Precision": np.mean(fold_precision),
    "Recall": np.mean(fold_recall),
    "F1": np.mean(fold_f1)
}

csv_path = os.path.join(SAVE_DIR, "KFold_Summary.csv")
results_df.to_csv(csv_path, index=False)

artifact = wandb.Artifact(
    "kfold-summary",
    type="results"
)

artifact.add_file(csv_path)

wandb.log_artifact(artifact)


  FOLD 1 / 5
  Freeze strategy   : last_4_blocks
  Trainable params  : 28,370,714 / 85,816,346 (33.1%)

Epoch 1/100  (LR: 2.00e-05)


Val: 100%|██████████| 195/195 [00:27<00:00,  6.98it/s]


Train Loss : 3.0449 | Val Loss  : 2.9109
Accuracy   : 0.2802  | Precision : 0.3392
Recall     : 0.2802  | F1 Score  : 0.2734
  ✓ Model saved (best val_f1: 0.2734) → models\model_fold_1.pth

Epoch 2/100  (LR: 4.00e-05)


Val: 100%|██████████| 195/195 [00:28<00:00,  6.90it/s]


Train Loss : 2.7457 | Val Loss  : 2.7029
Accuracy   : 0.3567  | Precision : 0.4219
Recall     : 0.3567  | F1 Score  : 0.3398
  ✓ Model saved (best val_f1: 0.3398) → models\model_fold_1.pth

Epoch 3/100  (LR: 6.00e-05)


...

Val: 100%|██████████| 195/195 [00:27<00:00,  6.99it/s]


Train Loss : 1.3889 | Val Loss  : 2.4061
Accuracy   : 0.5472  | Precision : 0.5749
Recall     : 0.5472  | F1 Score  : 0.5502

Epoch 19/100  (LR: 9.55e-05)


Val: 100%|██████████| 195/195 [00:27<00:00,  6.97it/s]


Train Loss : 1.3705 | Val Loss  : 2.3436
Accuracy   : 0.5578  | Precision : 0.5837
Recall     : 0.5578  | F1 Score  : 0.5605
Early Stopping Triggered
✓ Loss curve saved → models\Fold_2_Loss_Curve.png

Classification Report
                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.72      0.63      0.67       168
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.59      0.56      0.57       230
                                          Atopic Dermatitis Photos       0.42      0.70      0.52        98
                                            Bullous Disease Photos       0.25      0.66      0.37        89
                Cellulitis Impetigo and other Bacterial Infections       0.27      0.33      0.30        58
                                                     Eczema Photos       0.61      0.62      0.61       247
                    

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\UNIDA\\AppData\\Local\\Temp\\tmphwdx16v8wandb-media\\ko0hdf6x.table.json'

# **Grafik Gabungan & Final Summary**

In [ ]:
# ── GRAFIK GABUNGAN SEMUA FOLD ────────────────────────────────────────────────
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, fold_n in enumerate(all_train_losses.keys()):
    ep = range(1, len(all_train_losses[fold_n]) + 1)
    c  = colors[(fold_n - 1) % len(colors)]
    axes[0].plot(ep, all_train_losses[fold_n], label=f'Fold {fold_n}', color=c, marker='o', markersize=3)
    axes[1].plot(ep, all_val_losses[fold_n],   label=f'Fold {fold_n}', color=c, marker='o', markersize=3)

for ax, title in zip(axes, ['Train Loss — Semua Fold', 'Val Loss — Semua Fold']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Perbandingan Loss Semua Fold', fontsize=14, fontweight='bold')
plt.tight_layout()

combined_path = os.path.join(SAVE_DIR, "All_Folds_Loss_Curve.png")
fig.savefig(combined_path, dpi=150, bbox_inches="tight")
wandb.log({
    "Loss_Curve/All_Folds_Combined": wandb.Image(combined_path)
})
plt.close(fig)
print(f"✓ Grafik gabungan disimpan → {combined_path}")

# ── SUMMARY METRICS ───────────────────────────────────────────────────────────
print("\n" + "="*50)
print("  FINAL RESULT — ALL FOLDS")
print("="*50)
print(f"Mean Accuracy  : {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")
print(f"Mean Precision : {np.mean(fold_precision):.4f} ± {np.std(fold_precision):.4f}")
print(f"Mean Recall    : {np.mean(fold_recall):.4f} ± {np.std(fold_recall):.4f}")
print(f"Mean F1 Score  : {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")

# ── WANDB LOG SUMMARY ─────────────────────────────────────────────────────────
# Panel Summary → summary/mean_accuracy, summary/mean_precision, dst.
wandb.log({
    "summary/mean_accuracy"  : np.mean(fold_accuracies),
    "summary/mean_precision" : np.mean(fold_precision),
    "summary/mean_recall"    : np.mean(fold_recall),
    "summary/mean_f1"        : np.mean(fold_f1),
    "summary/std_accuracy"   : np.std(fold_accuracies),
    "summary/std_f1"         : np.std(fold_f1),
})



# **Test Evaluation**

In [ ]:
best_overall_path = all_fold_best_paths[fold_f1.index(max(fold_f1))]
print(f"Best model path : {best_overall_path}")
print(f"Best F1         : {max(fold_f1):.4f}")

test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_tf)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=num_classes
).to(device)


if USE_ECA:
    model.patch_embed = ECAPatchEmbed(
        model.patch_embed,
        eca_k_size=ECA_K_SIZE
    ).to(device)

checkpoint = torch.load(best_overall_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


# **Test Confusion Matrix**

In [ ]:
y_true, y_pred = [], []

with torch.no_grad():
    for images, lbs in tqdm(test_loader, desc="Test"):
        images  = images.to(device)
        outputs = model(images)
        y_true.extend(lbs.cpu().numpy())
        y_pred.extend(outputs.argmax(1).cpu().numpy())

acc       = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(15, 15))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
).plot(
    cmap="Blues",
    ax=ax,
    xticks_rotation=90
)

plt.tight_layout()

cm_path = os.path.join(SAVE_DIR, "Test_Confusion_Matrix.png")
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()

wandb.log({
    "Test/Confusion_Matrix": wandb.Image(cm_path),
    "test/accuracy": acc,
    "test/precision": precision,
    "test/recall": recall,
    "test/f1": f1
})

# ==========================================
# TEST RESULT CSV
# ==========================================

test_results_df = pd.DataFrame({
    "Filename": [test_dataset.samples[i][0] for i in range(len(y_true))],
    "True_Label": [classes[i] for i in y_true],
    "Predicted_Label": [classes[i] for i in y_pred],
    "Correct": np.array(y_true) == np.array(y_pred)
})

test_summary_df = pd.DataFrame([{
    "Accuracy": acc,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
}])

# =========================
# SIMPAN CSV
# =========================
summary_path = os.path.join(SAVE_DIR, "Test_Summary.csv")
test_summary_df.to_csv(summary_path, index=False)

csv_test_path = os.path.join(SAVE_DIR, "Test_Result.csv")
test_results_df.to_csv(csv_test_path, index=False)

print(f"Test CSV saved -> {csv_test_path}")

# =========================
# UPLOAD KE WANDB
# =========================
artifact = wandb.Artifact(
    name="test-results",
    type="results"
)

artifact.add_file(csv_test_path)
artifact.add_file(summary_path)

wandb.log_artifact(artifact)

wandb.finish()
print("\nWandB run selesai.")